In [1]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [2]:
print(f"Using the PyTorch version: {torch.__version__}")

## getting the device
device = "cuda" if torch.cuda.is_available() else 'cpu'
print(f"Device:{device}")

Using the PyTorch version: 2.8.0+cu128
Device:cuda


In [3]:
## Reading the Data

df = pd.read_csv('/home/prashant/Documents/ML-DL/PyTorch/100_Unique_QA_Dataset.csv')
print('-'*5,'First 10 rows','-'*5,'\n')
display(df.head(10))
print(f'Shape of the Dataset:{df.shape}')

----- First 10 rows ----- 



,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100
5,Who painted the Mona Lisa?,Leonardo-da-Vinci
6,What is the square root of 64?,8
7,What is the chemical symbol for gold?,Au
8,Which year did World War II end?,1945
9,What is the longest river in the world?,Nile


Shape of the Dataset:(90, 2)


In [21]:
# Tokenization

def tokenize(text):
    text = text.lower()
    text = text.replace('?','')
    text = text.replace("'","")

    tokens = text.split()

    return tokens

In [22]:
tokenize('What is the name of longest river in the world?')

['what', 'is', 'the', 'name', 'of', 'longest', 'river', 'in', 'the', 'world']

In [27]:
# Creating a vocabulary

vocab = {'<UNK>':0}

In [33]:
def update_vocab(row):

    question, answer = tokenize(row['question']), tokenize(row['answer'])

    qa = question + answer

    for word in qa: 
        if word not in vocab:
            vocab[word] = len(vocab)

    return row

In [34]:
df.apply(update_vocab, axis=1)

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100
...,...,...
85,Who directed the movie 'Titanic'?,JamesCameron
86,Which superhero is also known as the Dark Knight?,Batman
87,What is the capital of Brazil?,Brasilia
88,Which fruit is known as the king of fruits?,Mango


In [43]:
vocab

{'<UNK>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'germany': 8,
 'berlin': 9,
 'who': 10,
 'wrote': 11,
 'to': 12,
 'kill': 13,
 'a': 14,
 'mockingbird': 15,
 'harper-lee': 16,
 'largest': 17,
 'planet': 18,
 'in': 19,
 'our': 20,
 'solar': 21,
 'system': 22,
 'jupiter': 23,
 'boiling': 24,
 'point': 25,
 'water': 26,
 'celsius': 27,
 '100': 28,
 'painted': 29,
 'mona': 30,
 'lisa': 31,
 'leonardo-da-vinci': 32,
 'square': 33,
 'root': 34,
 '64': 35,
 '8': 36,
 'chemical': 37,
 'symbol': 38,
 'for': 39,
 'gold': 40,
 'au': 41,
 'which': 42,
 'year': 43,
 'did': 44,
 'world': 45,
 'war': 46,
 'ii': 47,
 'end': 48,
 '1945': 49,
 'longest': 50,
 'river': 51,
 'nile': 52,
 'japan': 53,
 'tokyo': 54,
 'developed': 55,
 'theory': 56,
 'relativity': 57,
 'albert-einstein': 58,
 'freezing': 59,
 'fahrenheit': 60,
 '32': 61,
 'known': 62,
 'as': 63,
 'red': 64,
 'mars': 65,
 'author': 66,
 '1984': 67,
 'george-orwell': 68,
 'currency': 69,
 'unit

In [41]:
def find_index(sentence, vocab):

    indexes = []

    for token in tokenize(sentence):
        if token in vocab:
            indexes.append(vocab[token])
        else:
            indexes.append(vocab['<UNK>'])
    return indexes

In [44]:
find_index('Paris is the capital of France.', vocab)

[7, 2, 3, 4, 5, 0]

In [45]:
## Creating the custom dataset class

class CustomDataset(Dataset):

    def __init__(self, df, vocab):
        self.df = df
        self.vocab = vocab

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):

        question = find_index(self.df.iloc[index]['question'], self.vocab)
        answer = find_index(self.df.iloc[index]['answer'], self.vocab)

        return torch.tensor(question), torch.tensor(answer)

In [46]:
data = CustomDataset(df, vocab)

In [72]:
dataloader = DataLoader(data, batch_size=1, shuffle=True)

In [61]:
len_list = []
for i in range(len(data)):
    q, a = data[i]
    question_len = len(q.tolist())
    len_list.append(question_len)

print(f"Max number of tokens available in questions: {max(len_list)}")

Max number of tokens available in questions: 12


In [62]:
# Check the data
for i in range(5):
    question, answer = data[i]
    print(f"Question Tensor: {question}")
    print(f"Answer Tensor: {answer}")
    print('-'*20)

Question Tensor: tensor([1, 2, 3, 4, 5, 6])
Answer Tensor: tensor([7])
--------------------
Question Tensor: tensor([1, 2, 3, 4, 5, 8])
Answer Tensor: tensor([9])
--------------------
Question Tensor: tensor([10, 11, 12, 13, 14, 15])
Answer Tensor: tensor([16])
--------------------
Question Tensor: tensor([ 1,  2,  3, 17, 18, 19, 20, 21, 22])
Answer Tensor: tensor([23])
--------------------
Question Tensor: tensor([ 1,  2,  3, 24, 25,  5, 26, 19, 27])
Answer Tensor: tensor([28])
--------------------


In [85]:
## Model Building

class SimpleRNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=50)
        self.rnn = nn.RNN(50, 64,batch_first=True)
        self.fc = nn.Linear(64, vocab_size)

    def forward(self, question):
        embeded_question = self.embedding(question)
        hidden, final = self.rnn(embeded_question)
        output = self.fc(final.squeeze(0))
        
        return output

In [86]:
learning_rate = 0.001
epochs = 20

model = SimpleRNN(len(vocab)).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
for epoch in range(epochs):

    total_loss = 0

    for ques, ans in dataloader:

        ques, ans = ques.to(device), ans.to(device)

        optimizer.zero_grad()

        # prediction
        y_pred = model(ques)

        # loss
        loss = criterion(y_pred, ans[0])

        # backpropagation
        loss.backward()

        # update the weights
        optimizer.step()

        ## Update the total loss
        total_loss += loss.item()

    print(f"Epoch: {epoch+1:<2}/{epochs}, Loss: {total_loss:4f}")


Epoch: 1 /20, Loss: 0.525493
Epoch: 2 /20, Loss: 0.498792
Epoch: 3 /20, Loss: 0.473763
Epoch: 4 /20, Loss: 0.449913
Epoch: 5 /20, Loss: 0.427259
Epoch: 6 /20, Loss: 0.406031
Epoch: 7 /20, Loss: 0.385618
Epoch: 8 /20, Loss: 0.366851
Epoch: 9 /20, Loss: 0.348844
Epoch: 10/20, Loss: 0.332101
Epoch: 11/20, Loss: 0.315897
Epoch: 12/20, Loss: 0.300285
Epoch: 13/20, Loss: 0.285935
Epoch: 14/20, Loss: 0.272147
Epoch: 15/20, Loss: 0.259249
Epoch: 16/20, Loss: 0.246903
Epoch: 17/20, Loss: 0.235020
Epoch: 18/20, Loss: 0.223800
Epoch: 19/20, Loss: 0.213397
Epoch: 20/20, Loss: 0.203430


In [106]:
def predict(model=model, question=' ', thresold=0.5):

    question_array = find_index(question, vocab)
    question_tensor = torch.tensor(question_array).unsqueeze(0).to(device)

    output = model(question_tensor)

    probabilities = torch.nn.functional.softmax(output, dim=1)

    value, index = torch.max(probabilities, dim=1)

    print(f"Question: {question}", \
          f"Answer: {list(vocab.keys())[index.item()].upper() if value.item() > thresold else "I don't know"}", \
            sep='\n')

In [108]:
def get_question():
    question = input("Ask a question: ")
    return question

predict(question=get_question())

Question: How many animals are carnivores?
Answer: I don't know
